# 07 — American Options

Price American options using multiple methods:
- **Barone-Adesi-Whaley** analytic approximation
- **Bjerksund-Stensland** approximation
- **Finite Differences** (implicit Euler, Crank-Nicolson)
- **Binomial trees** (CRR, JR, Tian, Trigeorgis, Leisen-Reimer)
- **Monte Carlo** (Longstaff-Schwartz LSM)
- American Greeks via AD

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, load_quantlib, compare_table, timed_ms
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np
QL = load_quantlib()

In [ ]:
# American put parameters (classic test case)
S, K, T = 36.0, 40.0, 1.0
r, q, sigma = 0.06, 0.0, 0.20
PUT = -1

---
## 1. QuantLib Reference

In [ ]:
ql_today = QL.Date(15, 1, 2024)
QL.Settings.instance().evaluationDate = ql_today

ql_spot   = QL.QuoteHandle(QL.SimpleQuote(S))
ql_r_ts   = QL.YieldTermStructureHandle(QL.FlatForward(ql_today, r, QL.Actual365Fixed()))
ql_q_ts   = QL.YieldTermStructureHandle(QL.FlatForward(ql_today, q, QL.Actual365Fixed()))
ql_vol    = QL.BlackVolTermStructureHandle(QL.BlackConstantVol(ql_today, QL.NullCalendar(), sigma, QL.Actual365Fixed()))
ql_proc   = QL.BlackScholesMertonProcess(ql_spot, ql_q_ts, ql_r_ts, ql_vol)

exercise = QL.AmericanExercise(ql_today, ql_today + QL.Period(1, QL.Years))
payoff   = QL.PlainVanillaPayoff(QL.Option.Put, K)
ql_opt   = QL.VanillaOption(payoff, exercise)

ql_prices = {}

# Analytic approximations
ql_opt.setPricingEngine(QL.BaroneAdesiWhaleyApproximationEngine(ql_proc))
ql_prices["BAW"] = ql_opt.NPV()

ql_opt.setPricingEngine(QL.BjerksundStenslandApproximationEngine(ql_proc))
ql_prices["Bjerksund-Stensland"] = ql_opt.NPV()

# FD
ql_opt.setPricingEngine(QL.FdBlackScholesVanillaEngine(ql_proc, 800, 200))
ql_prices["FD"] = ql_opt.NPV()

# Binomial
for name in ["crr", "jr", "tian", "trigeorgis", "lr"]:
    ql_opt.setPricingEngine(QL.BinomialVanillaEngine(ql_proc, name, 801))
    ql_prices[f"Binomial ({name})"] = ql_opt.NPV()

for name, price in ql_prices.items():
    print(f"QL {name:25s}: {price:.8f}")

---
## 2. ql-jax Multi-Engine Pricing

In [ ]:
from ql_jax.engines.analytic.american import barone_adesi_whaley_price, bjerksund_stensland_price
from ql_jax.engines.fd.black_scholes import fd_black_scholes_price
from ql_jax.engines.lattice.binomial import binomial_price
from ql_jax.engines.mc.american import mc_american_bs

jax_prices = {}

jax_prices["BAW"] = float(barone_adesi_whaley_price(S, K, T, r, q, sigma, PUT))
jax_prices["Bjerksund-Stensland"] = float(bjerksund_stensland_price(S, K, T, r, q, sigma, PUT))

jax_prices["FD"] = float(fd_black_scholes_price(
    S, K, T, r, q, sigma, PUT, n_space=800, n_time=200, american=True))

for name in ["crr", "jr", "tian", "trigeorgis", "leisen_reimer"]:
    jax_prices[f"Binomial ({name})"] = float(binomial_price(
        S, K, T, r, q, sigma, PUT, n_steps=801, tree_type=name, american=True))

mc_price = float(mc_american_bs(
    S, K, T, r, q, sigma, PUT, n_paths=50_000, n_steps=50,
    key=jax.random.PRNGKey(42)))
jax_prices["MC (LSM)"] = mc_price

for name, price in jax_prices.items():
    print(f"JAX {name:25s}: {price:.8f}")

In [ ]:
# Comparison table
rows = []
for name in ["BAW", "Bjerksund-Stensland", "FD"]:
    rows.append((name, ql_prices[name], jax_prices[name]))
for name in ["crr", "jr", "tian", "trigeorgis"]:
    rows.append((f"Binomial ({name})", ql_prices[f"Binomial ({name})"],
                 jax_prices[f"Binomial ({name})"]))

compare_table(rows, title="American Put (S=36, K=40, σ=20%, r=6%, T=1Y)")

---
## 3. Early Exercise Premium

The American premium = American price − European price.

In [ ]:
from ql_jax.engines.analytic.black_formula import black_scholes_price

euro_price = float(black_scholes_price(S, K, T, r, q, sigma, PUT))
amer_price = jax_prices["FD"]
early_premium = amer_price - euro_price

print(f"European put : {euro_price:.6f}")
print(f"American put : {amer_price:.6f}")
print(f"Early exercise premium: {early_premium:.6f} ({early_premium/euro_price*100:.2f}%)")

---
## 4. American Greeks via AD

In [ ]:
def amer_put_price(spot, vol):
    return barone_adesi_whaley_price(spot, K, T, r, q, vol, PUT)

s0, v0 = jnp.float64(S), jnp.float64(sigma)

delta = float(jax.grad(amer_put_price, argnums=0)(s0, v0))
vega  = float(jax.grad(amer_put_price, argnums=1)(s0, v0))
gamma = float(jax.grad(jax.grad(amer_put_price, argnums=0), argnums=0)(s0, v0))

print("American Put Greeks (BAW + AD):")
print(f"  Delta : {delta:+.6f}")
print(f"  Vega  : {vega:+.6f}")
print(f"  Gamma : {gamma:+.6f}")

In [ ]:
# Delta surface: American vs European
import matplotlib.pyplot as plt

spots = jnp.linspace(20, 60, 100)

amer_deltas = jax.vmap(lambda s: jax.grad(amer_put_price, argnums=0)(s, v0))(spots)
euro_deltas = jax.vmap(lambda s: jax.grad(
    lambda s_: black_scholes_price(s_, K, T, r, q, sigma, PUT))(s))(spots)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.array(spots), np.array(amer_deltas), 'b-', label='American (BAW)', linewidth=2)
ax.plot(np.array(spots), np.array(euro_deltas), 'r--', label='European (BSM)', linewidth=2)
ax.axvline(K, color='k', linestyle=':', alpha=0.5, label=f'Strike={K}')
ax.set_xlabel('Spot')
ax.set_ylabel('Delta')
ax.set_title('American vs European Put Delta')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

---
## 5. Exercises

1. **Convergence**: Plot FD price vs `n_space` (50–1000) for both implicit Euler (`theta_fd=1.0`) and Crank-Nicolson (`theta_fd=0.5`).
2. **American call with dividends**: Set $q=0.08$ and verify the American call premium is nonzero.
3. **LSM convergence**: Vary `n_paths` from 1,000 to 200,000 and plot MC price convergence.

---
*End of Notebook 07*